# **Extração e estruturação dos dados de Warner et al. (2022) [1]**



Os dados experimentais de toxicidade de crioprotetores (CPAs) utilizados neste projeto provêm dos arquivos suplementares de Warner et al. (2022), disponibilizados como planilhas `.xls`.  Cada planilha segue um formato semi-estruturado: as linhas iniciais contêm metadados do experimento, e os dados em si estão organizados em blocos — um por composição de CPA — sem um esquema tabular convencional. Este notebook converte esses arquivos para o formato utilizado pelo notebook principal (`Academico.ipynb`), em que cada linha representa uma única observação em um tempo de exposição específico, para uma composição definida.

## **Funcionamento**

O **.csv** bruto é lido pulando as 15 primeiras linhas de metadados (``skiprows=15``) e mantendo apenas as quatro primeiras colunas, que contêm espectivamente o rótulo da composição (ou a concentração máxima), o tempo de exposição, a temperatura e a viabilidade celular. 

A coluna de rótulos funciona como marcador dos bloco seguinte, quando seu valor coincide com um dos nomes de CPA conhecidos (ex: ``"Gly"``, ``"DMSO + PG"``), ela sinaliza o início de um novo grupo e a composição ativa é atualizada; quando seu valor é ``"Max Conc"``, a linha é ignorada; nas demais linhas, o valor é a concentração máxima do experimento.

A concentração de cada CPA base na mistura é calculada dividindo a concentração máxima pelo número de componentes presentes, assumindo equimolaridade, conforme foi descrito no artigo. CPAs ausentes recebem concentração zero. O controle positivo (``"Pos Con"``) é registrado com todos os CPAs em zero; o controle negativo (``"Neg Con"``) é descartado, pois corresponde a células sem exposição e não entra no ajuste da PINN. 
O resultado é salvo como *.csv* em ``dados/dados_simples_e_binarios.csv``.

O mesmo procedimento é aplicado aos dados ternários (``Ternary-Data-Report.csv``), com a lista de composições atualizada para as dez misturas de três componentes do artigo, produzindo ``dados/dados_ternarios.csv``.

In [4]:
import pandas as pd
import numpy as np

## **Dados simples e binários**

In [5]:
df = pd.read_csv("dados/Single-and-Binary-Data-Report.csv", skiprows=15)
df = df.iloc[:, :4]
df = df.dropna(how='all')

CPAs = ["Gly", "DMSO", "PG", "EG", "FA", "Gly + DMSO", "Gly + PG", "Gly + EG", "Gly + FA",
        "DMSO + PG", "DMSO + EG", "DMSO + FA", "PG + EG", "PG + FA", "EG + FA", "Pos Con", "Neg Con"]

bases = ["Gly", "DMSO", "PG", "EG", "FA"]

dados = {
    "Gly": [],
    "DMSO": [],
    "PG": [],
    "EG": [],
    "FA": [],
    "Exp Time (min)": [],
    "Temp (°C)": [],
    "Cell Viab (%)": [],
}

for row in df.itertuples():
    valor_coluna = str(row[1])
    
    if valor_coluna in CPAs:
        cpas = valor_coluna.split(" + ")
        continue

    if valor_coluna == "Max Conc":
        continue

    len_cpas = len(cpas)
    max_conc = float(valor_coluna)
    
    if "Pos Con" in cpas:
        for c_cpa in bases:
            dados[c_cpa].append(0)
            
    elif "Neg Con" in cpas:
        continue
        for c_cpa in bases:
            dados[c_cpa].append(None)
            
    else:
        for c_cpa in bases:
            if c_cpa in cpas:
                dados[c_cpa].append(max_conc / len_cpas)
            else:
                dados[c_cpa].append(0)

    dados["Exp Time (min)"].append(row[2])
    dados["Temp (°C)"].append(row[3])
    dados["Cell Viab (%)"].append(row[4])


df_limpo = pd.DataFrame(dados)
df_limpo = df_limpo.rename(columns={"Gly": "Gly (mol/L)", "DMSO": "DMSO (mol/L)", "PG": "PG (mol/L)", "EG": "EG (mol/L)", "FA": "FA (mol/L)"})
df_limpo.to_csv('dados/dados_simples_e_binarios.csv', index=False)
display(df_limpo)

,Gly (mol/L),DMSO (mol/L),PG (mol/L),EG (mol/L),FA (mol/L),Exp Time (min),Temp (°C),Cell Viab (%)
0,0.0,0.0,0.0,0.0,0.0,5,23.9,1.0672563952969
1,0.0,0.0,0.0,0.0,0.0,5,23.9,0.97670087233233
2,0.0,0.0,0.0,0.0,0.0,5,23.9,0.95431161208384
3,0.0,0.0,0.0,0.0,0.0,5,23.9,0.91876820807815
4,0.0,0.0,0.0,0.0,0.0,5,23.9,1.0829629122088
...,...,...,...,...,...,...,...,...
1615,0.0,0.0,0.0,5.0,5.0,60,27.1,0.0032896970420634
1616,0.0,0.0,0.0,5.0,5.0,60,27.1,0.0030693801377293
1617,0.0,0.0,0.0,5.0,5.0,60,27.1,0.00093349308449569
1618,0.0,0.0,0.0,5.0,5.0,60,27.1,-0.005173134689141


## **Dados ternários**

In [6]:
df = pd.read_csv("dados/Ternary-Data-Report.csv", skiprows=15)
df = df.iloc[:, :4]
df = df.dropna(how='all')

CPAs = ["Pos Con", "Neg Con", "Gly + DMSO + PG", "Gly + DMSO + EG", "Gly + DMSO + FA", 
        "Gly + PG + EG", "Gly + PG + FA", "Gly + EG + FA", "DMSO + PG + EG", 
        "DMSO + PG + FA", "DMSO + EG + FA", "PG + EG + FA"]

bases = ["Gly", "DMSO", "PG", "EG", "FA"]

dados = {
    "Gly": [],
    "DMSO": [],
    "PG": [],
    "EG": [],
    "FA": [],
    "Exp Time (min)": [],
    "Temp (°C)": [],
    "Cell Viab (%)": [],
}

for row in df.itertuples():
    valor_coluna = str(row[1])
    
    if valor_coluna in CPAs:
        cpas = valor_coluna.split(" + ")
        continue

    if valor_coluna == "Max Conc":
        continue

    len_cpas = len(cpas)
    max_conc = float(valor_coluna)
    
    if "Pos Con" in cpas:
        for c_cpa in bases:
            dados[c_cpa].append(0)
            
    elif "Neg Con" in cpas:
        continue
        for c_cpa in bases:
            dados[c_cpa].append(None)
            
    else:
        for c_cpa in bases:
            if c_cpa in cpas:
                dados[c_cpa].append(max_conc / len_cpas)
            else:
                dados[c_cpa].append(0)

    dados["Exp Time (min)"].append(row[2])
    dados["Temp (°C)"].append(row[3])
    dados["Cell Viab (%)"].append(row[4])


df_limpo = pd.DataFrame(dados)
df_limpo = df_limpo.rename(columns={"Gly": "Gly (mol/L)", "DMSO": "DMSO (mol/L)", "PG": "PG (mol/L)", "EG": "EG (mol/L)", "FA": "FA (mol/L)"})
df_limpo.to_csv('dados/dados_ternarios.csv', index=False)
display(df_limpo)

,Gly (mol/L),DMSO (mol/L),PG (mol/L),EG (mol/L),FA (mol/L),Exp Time (min),Temp (°C),Cell Viab (%)
0,0.0,0.0,0.000000,0.000000,0.000000,5,25.75,0.91194671087964
1,0.0,0.0,0.000000,0.000000,0.000000,5,25.75,1.0160813096894
2,0.0,0.0,0.000000,0.000000,0.000000,5,25.75,1.0371753854127
3,0.0,0.0,0.000000,0.000000,0.000000,5,25.75,1.0501721440171
4,0.0,0.0,0.000000,0.000000,0.000000,5,25.75,0.98462445000116
...,...,...,...,...,...,...,...,...
218,0.0,0.0,2.333333,2.333333,2.333333,60,25.5,0.25572648389472
219,0.0,0.0,2.333333,2.333333,2.333333,60,25.5,0.012377890116404
220,0.0,0.0,2.333333,2.333333,2.333333,60,25.5,0.22418508768892
221,0.0,0.0,2.333333,2.333333,2.333333,60,25.5,0.24500357224883


## **Referências**

[1] Warner, R. M., et al. (2022). Rapid Quantification of Cryoprotectant Permeability via Numerical Optimization of Differential Scanning Calorimetry Thermograms. Cryobiology. DOI: 10.1016.

**Autor**: Arthur Brandão do Nascimento